In [30]:
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from flax.core import unfreeze, freeze

from pyscf import gto, scf, ao2mo, cc

import importlib

import wavefunctions, hamiltonian, trajectory, qc

import time

## 3D Cubic Lattice

In [36]:
importlib.reload(wavefunctions)
importlib.reload(hamiltonian)

spins = (7,7)
dim = 3

N = spins[0] + spins[1]

for r_ws in [1.0, 2.0, 5.0, 10.0, 20.0, 50.0]:

    # cubic lattice
    lattice = wavefunctions.computeLattice(N, r_ws, dim)
    
    system = qc.ueg_qc(0, spins, dim=dim, e_cut_red=10.0/(r_ws**2), basis=lattice)
    k_points = system.get_k_points()
    n_kpts = k_points.shape[0]
    print(r_ws, n_kpts)
    h1 = system.get_h1(k_points)
    eri_jax = system.get_eri_tensor_real(k_points)
    eri = np.asarray(eri_jax, dtype=np.double)

    mol = gto.M(verbose=0)
    mol.nelectron = system.n_particles
    mol.incore_anyway = True
    mol.energy_nuc = lambda *args: 0.0
    mol.verbose = 0

    uhf_energy = np.inf
    best_umf = None

    umf = scf.UHF(mol)
    umf.max_cycle = 200
    umf.get_hcore = lambda *args: [h1, h1]
    umf.get_ovlp = lambda *args: np.eye(n_kpts)
    umf._eri = ao2mo.restore(8, eri, n_kpts)
    # umf.init_guess = "1e"
    start = time.time()
    for seed in range(3):
        np.random.seed(seed)
        dm0 = umf.get_init_guess()
        dm0[0] += np.random.randn(n_kpts, n_kpts) * 0.1
        dm0[1] += np.random.randn(n_kpts, n_kpts) * 0.1
        escf = umf.kernel(dm0)
        mo1 = umf.stability()[0]
        if uhf_energy is np.inf or escf < uhf_energy:
            uhf_energy = escf
            best_umf = umf

    mycc = cc.CCSD(best_umf)
    ecc = mycc.kernel()
    e_t = mycc.ccsd_t()
    ccsd_energy = mycc.e_tot
    ccsdt_energy = mycc.e_tot + e_t

    print(f"UHF:     \t{uhf_energy / N}")
    print(f"CCSD:    \t{ccsd_energy / N}")
    print(f"CCSD(T): \t{ccsdt_energy / N}")

    print()

1.0 81
UHF:     	0.6065345994595975
CCSD:    	0.572209101146145
CCSD(T): 	0.570908853964825

2.0 81
UHF:     	0.02303901700811422
CCSD:    	-0.004502618924811935
CCSD(T): 	-0.007557431457124139

5.0 81
UHF:     	-0.059336991808130846
CCSD:    	-0.07537580602899184
CCSD(T): 	-0.08173105411489033

10.0 81
UHF:     	-0.04472906625242317
CCSD:    	-0.05112114863174277
CCSD(T): 	-0.05488391755280825

20.0 81
UHF:     	-0.027546777313485038
CCSD:    	-0.029773858272937437
CCSD(T): 	-0.030482519100723672

50.0 81
UHF:     	-0.012688642472718932
CCSD:    	-0.013363274123352994
CCSD(T): 	-0.013527104635752971



## 3D Skewed Lattice

In [ ]:
importlib.reload(wavefunctions)
importlib.reload(hamiltonian)

spins = (7,7)
dim = 3

N = spins[0] + spins[1]

for r_ws in [1.0, 2.0, 5.0, 10.0, 20.0, 50.0]:

    # non-cubic lattice
    lattice = wavefunctions.computeLattice(
        N, r_ws, dim,
        basis_matrix=jnp.eye(dim) + 0.2
    )
    
    system = qc.ueg_qc(0, spins, dim=dim, e_cut_red=10.0/(r_ws**2), basis=lattice)
    k_points = system.get_k_points()
    n_kpts = k_points.shape[0]
    print(r_ws, n_kpts)
    h1 = system.get_h1(k_points)
    eri_jax = system.get_eri_tensor_real(k_points)
    eri = np.asarray(eri_jax, dtype=np.double)

    mol = gto.M(verbose=0)
    mol.nelectron = system.n_particles
    mol.incore_anyway = True
    mol.energy_nuc = lambda *args: 0.0
    mol.verbose = 0

    uhf_energy = np.inf
    best_umf = None

    umf = scf.UHF(mol)
    umf.max_cycle = 200
    umf.get_hcore = lambda *args: [h1, h1]
    umf.get_ovlp = lambda *args: np.eye(n_kpts)
    umf._eri = ao2mo.restore(8, eri, n_kpts)
    # umf.init_guess = "1e"
    start = time.time()
    for seed in range(3):
        np.random.seed(seed)
        dm0 = umf.get_init_guess()
        dm0[0] += np.random.randn(n_kpts, n_kpts) * 0.1
        dm0[1] += np.random.randn(n_kpts, n_kpts) * 0.1
        escf = umf.kernel(dm0)
        mo1 = umf.stability()[0]
        if uhf_energy is np.inf or escf < uhf_energy:
            uhf_energy = escf
            best_umf = umf

    mycc = cc.CCSD(best_umf)
    ecc = mycc.kernel()
    e_t = mycc.ccsd_t()
    ccsd_energy = mycc.e_tot
    ccsdt_energy = mycc.e_tot + e_t

    print(f"UHF:     \t{uhf_energy / N}")
    print(f"CCSD:    \t{ccsd_energy / N}")
    print(f"CCSD(T): \t{ccsdt_energy / N}")

    print()

1.0 89


/home/amress/miniforge3/envs/nqs/lib/python3.9/site-packages/pyscf/gto/mole.py:1293: UserWarning: Function mol.dumps drops attribute energy_nuc because it is not JSON-serializable
  warnings.warn(msg)


UHF:     	0.7106951060045711
CCSD:    	0.6751886755878669
CCSD(T): 	0.6736644686327355

2.0 89


## 2D Square Lattice

In [37]:
importlib.reload(wavefunctions)
importlib.reload(hamiltonian)

spins = (5,5)
dim = 2

N = spins[0] + spins[1]

for r_ws in [1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0]:

    # square lattice
    lattice = wavefunctions.computeLattice(N, r_ws, dim)
    
    system = qc.ueg_qc(0, spins, dim=dim, e_cut_red=10.0/(r_ws**2), basis=lattice)
    k_points = system.get_k_points()
    n_kpts = k_points.shape[0]
    print(r_ws, n_kpts)
    h1 = system.get_h1(k_points)
    eri_jax = system.get_eri_tensor_real(k_points)
    eri = np.asarray(eri_jax, dtype=np.double)

    mol = gto.M(verbose=0)
    mol.nelectron = system.n_particles
    mol.incore_anyway = True
    mol.energy_nuc = lambda *args: 0.0
    mol.verbose = 0

    uhf_energy = np.inf
    best_umf = None

    umf = scf.UHF(mol)
    umf.max_cycle = 200
    umf.get_hcore = lambda *args: [h1, h1]
    umf.get_ovlp = lambda *args: np.eye(n_kpts)
    umf._eri = ao2mo.restore(8, eri, n_kpts)
    # umf.init_guess = "1e"
    start = time.time()
    for seed in range(3):
        np.random.seed(seed)
        dm0 = umf.get_init_guess()
        dm0[0] += np.random.randn(n_kpts, n_kpts) * 0.1
        dm0[1] += np.random.randn(n_kpts, n_kpts) * 0.1
        escf = umf.kernel(dm0)
        mo1 = umf.stability()[0]
        if uhf_energy is np.inf or escf < uhf_energy:
            uhf_energy = escf
            best_umf = umf

    mycc = cc.CCSD(best_umf)
    ecc = mycc.kernel()
    e_t = mycc.ccsd_t()
    ccsd_energy = mycc.e_tot
    ccsdt_energy = mycc.e_tot + e_t

    print(f"UHF:     \t{uhf_energy / N}")
    print(f"CCSD:    \t{ccsd_energy / N}")
    print(f"CCSD(T): \t{ccsdt_energy / N}")

    print()

1.0 45


/home/amress/miniforge3/envs/nqs/lib/python3.9/site-packages/pyscf/gto/mole.py:1293: UserWarning: Function mol.dumps drops attribute energy_nuc because it is not JSON-serializable
  warnings.warn(msg)


UHF:     	-0.12461069820167721
CCSD:    	-0.21707019750151307
CCSD(T): 	-0.22529965128280086

2.0 45
UHF:     	-0.201028966823817
CCSD:    	-0.25266668641630013
CCSD(T): 	-0.26627075012022516

5.0 45
UHF:     	-0.1325183134413146
CCSD:    	-0.14485786518532262
CCSD(T): 	-0.14850592437643664

10.0 45
UHF:     	-0.07899996057038243
CCSD:    	-0.08300138200151234
CCSD(T): 	-0.08419665489392315

20.0 45
UHF:     	-0.043762391568091516
CCSD:    	-0.04505545024336526
CCSD(T): 	-0.04530737762608891

50.0 45
UHF:     	-0.018709035741383356
CCSD:    	-0.01911102369590286
CCSD(T): 	-0.019172599665363763

100.0 45
UHF:     	-0.009563354178990211
CCSD:    	-0.009766692056083734
CCSD(T): 	-0.009793351182154167



## 2D Skewed Lattice

In [38]:
importlib.reload(wavefunctions)
importlib.reload(hamiltonian)

spins = (5,5)
dim = 2

N = spins[0] + spins[1]

for r_ws in [1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0]:

    # non-square lattice
    lattice = wavefunctions.computeLattice(
        N, r_ws, dim,
        basis_matrix=jnp.eye(dim) + 0.2
    )
    
    system = qc.ueg_qc(0, spins, dim=dim, e_cut_red=10.0/(r_ws**2), basis=lattice)
    k_points = system.get_k_points()
    n_kpts = k_points.shape[0]
    print(r_ws, n_kpts)
    h1 = system.get_h1(k_points)
    eri_jax = system.get_eri_tensor_real(k_points)
    eri = np.asarray(eri_jax, dtype=np.double)

    mol = gto.M(verbose=0)
    mol.nelectron = system.n_particles
    mol.incore_anyway = True
    mol.energy_nuc = lambda *args: 0.0
    mol.verbose = 0

    uhf_energy = np.inf
    best_umf = None

    umf = scf.UHF(mol)
    umf.max_cycle = 200
    umf.get_hcore = lambda *args: [h1, h1]
    umf.get_ovlp = lambda *args: np.eye(n_kpts)
    umf._eri = ao2mo.restore(8, eri, n_kpts)
    # umf.init_guess = "1e"
    start = time.time()
    for seed in range(3):
        np.random.seed(seed)
        dm0 = umf.get_init_guess()
        dm0[0] += np.random.randn(n_kpts, n_kpts) * 0.1
        dm0[1] += np.random.randn(n_kpts, n_kpts) * 0.1
        escf = umf.kernel(dm0)
        mo1 = umf.stability()[0]
        if uhf_energy is np.inf or escf < uhf_energy:
            uhf_energy = escf
            best_umf = umf

    mycc = cc.CCSD(best_umf)
    ecc = mycc.kernel()
    e_t = mycc.ccsd_t()
    ccsd_energy = mycc.e_tot
    ccsdt_energy = mycc.e_tot + e_t

    print(f"UHF:     \t{uhf_energy / N}")
    print(f"CCSD:    \t{ccsd_energy / N}")
    print(f"CCSD(T): \t{ccsdt_energy / N}")

    print()

1.0 51


/home/amress/miniforge3/envs/nqs/lib/python3.9/site-packages/pyscf/gto/mole.py:1293: UserWarning: Function mol.dumps drops attribute energy_nuc because it is not JSON-serializable
  warnings.warn(msg)


UHF:     	-0.10373934436340475
CCSD:    	-0.18966798976801244
CCSD(T): 	-0.2012662727074626

2.0 51
UHF:     	-0.20360081031852822
CCSD:    	-0.24879663727634393
CCSD(T): 	-0.25869541207413194

5.0 51
UHF:     	-0.1318231444590478
CCSD:    	-0.14483654427015136
CCSD(T): 	-0.14822565039999852

10.0 51
UHF:     	-0.07924255426211917
CCSD:    	-0.08316312739193378
CCSD(T): 	-0.0841846912825911

20.0 51
UHF:     	-0.04412430497827079
CCSD:    	-0.04540408096034569
CCSD(T): 	-0.04558974110283686

50.0 51
UHF:     	-0.018836417992695923
CCSD:    	-0.019126227290232955
CCSD(T): 	-0.019201326685024354

100.0 51
UHF:     	-0.009654876355949641
CCSD:    	-0.00984375607605292
CCSD(T): 	-0.009865358875173286

